In [7]:
import os
import pandas as pd
from typing import Dict, List, Tuple
from pathlib import Path
import sys
import contextlib
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import random
import torch
from PIL import Image

OUT_DIR = "/home/infres/yrothlin-24/CHAL_IM05/data_analysis/outputs"


label2id = {
    "SNE": 0, "LY": 1, "MO": 2, "EO": 3, "BA": 4, "VLY": 5, "BNE": 6, "MMY": 7, "MY": 8, "PMY": 9, "BL": 10, "PC": 11, "PLY": 12,
}

train_dir = "/tsi/data_education/ChallengeIMA205/IMA205-challenge/train"
train_label_path = "/tsi/data_education/ChallengeIMA205/IMA205-challenge/train_metadata.csv"

test_dir = "/tsi/data_education/ChallengeIMA205/IMA205-challenge/test"
test_label_path = "/tsi/data_education/ChallengeIMA205/IMA205-challenge/test_metadata.csv"

In [2]:
# Utils functions

def get_filespath(dataset_dir: str) -> List[str]:
    files = []
    for dirpath, _, filenames in os.walk(dataset_dir):
        for filename in filenames:
            if filename.endswith(".png"):
                files.append(os.path.join(dirpath, filename))
    files.sort()
    return files


def get_labels(csv_path: str) -> Dict[str, int]:
    df = pd.read_csv(csv_path)
    labels = {}
    for _, row in df.iterrows():
        labels[str(row["ID"])] = label2id[str(row["label"])]
    return labels


def get_base_id(filename: str) -> str:
    p = Path(filename)
    stem = p.stem
    suffix = p.suffix
    marker = "_aug_"
    if marker in stem:
        stem = stem.split(marker)[0]
    return f"{stem}{suffix}"

@contextlib.contextmanager
def _filter_stderr(substr: str):
    r_fd, w_fd = os.pipe()
    orig_fd = os.dup(2)
    os.dup2(w_fd, 2)
    os.close(w_fd)
    try:
        yield
    finally:
        os.dup2(orig_fd, 2)
        os.close(orig_fd)
        with os.fdopen(r_fd, "r") as r:
            for line in r:
                if substr not in line:
                    sys.stderr.write(line)

def imread_silent(path: str):
    with _filter_stderr("Corrupt JPEG data"):
        return cv2.imread(path, cv2.IMREAD_COLOR)


In [3]:
def save_label_distribution_histogram(dataset_dir: str, csv_path: str, out_dir: str, filename: str = "label_distribution.png"):
    os.makedirs(out_dir, exist_ok=True)

    labels_map = get_labels(csv_path)
    id2label = {v: k for k, v in label2id.items()}
    files = get_filespath(dataset_dir)

    counts = Counter()
    for filepath in files:
        base_id = get_base_id(filepath)
        if base_id in labels_map:
            counts[labels_map[base_id]] += 1

    sorted_items = sorted(counts.items())
    labels = [id2label[k] for k, _ in sorted_items]
    values = [v for _, v in sorted_items]

    plt.figure(figsize=(10, 6))
    plt.bar(labels, values)
    plt.xlabel("Labels")
    plt.ylabel("Count")
    plt.title("Label Distribution")
    plt.xticks(rotation=45)

    for i, v in enumerate(values):
        plt.text(i, v, str(v), ha='center', va='bottom')

    plt.tight_layout()

    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path)
    plt.close()



    

save_label_distribution_histogram(
    dataset_dir=train_dir,
    csv_path=train_label_path,
    out_dir=OUT_DIR
)

In [6]:
def save_class_samples_grid(class_id: int, dataset_dir: str, csv_path: str, out_dir: str, filename: str = None, n_samples: int = 5):
    os.makedirs(out_dir, exist_ok=True)

    labels_map = get_labels(csv_path)
    id2label = {v: k for k, v in label2id.items()}
    files = get_filespath(dataset_dir)

    selected_files = []
    for filepath in files:
        base_id = get_base_id(filepath)
        if base_id in labels_map and labels_map[base_id] == class_id:
            selected_files.append(filepath)

    if len(selected_files) == 0:
        raise ValueError(f"No samples found for class_id={class_id}")

    selected_files = random.sample(selected_files, min(n_samples, len(selected_files)))

    images = []
    for path in selected_files:
        img = imread_silent(path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)

    if len(images) == 0:
        raise ValueError("No valid images could be loaded")

    plt.figure(figsize=(15, 3))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(img)
        plt.axis("off")

    label_name = id2label[class_id]
    plt.suptitle(f"Class {label_name} ({class_id})")

    plt.tight_layout()

    if filename is None:
        filename = f"class_{label_name}_{class_id}.png"

    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path)
    plt.close()


for class_id in range(len(label2id)):
    save_class_samples_grid(
        class_id=class_id,
        dataset_dir=train_dir,
        csv_path=train_label_path,
        out_dir=OUT_DIR,
        n_samples=5
    )

In [10]:
class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8), p=0.5):
        self.p = p
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        if np.random.rand() > self.p:
            return img

        img_np = np.array(img)

        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size,
        )
        l = clahe.apply(l)

        lab = cv2.merge((l, a, b))
        out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

        return Image.fromarray(out)



class AddGaussianNoise(torch.nn.Module):
    def __init__(self, p=0.3, std_range=(0.005, 0.02)):
        super().__init__()
        self.p = p
        self.std_range = std_range

    def forward(self, x):
        if torch.rand(1).item() < self.p:
            std = torch.empty(1).uniform_(*self.std_range).item()
            noise = torch.randn_like(x) * std
            x = torch.clamp(x + noise, 0.0, 1.0)
        return x





def save_clahe_samples(dataset_dir: str, csv_path: str, out_dir: str, filename: str = "clahe_samples.png", n_samples: int = 5):
    os.makedirs(out_dir, exist_ok=True)

    files = get_filespath(dataset_dir)
    labels_map = get_labels(csv_path)
    id2label = {v: k for k, v in label2id.items()}

    selected_files = random.sample(files, min(n_samples, len(files)))
    transform = CLAHETransform(p=1.0)

    originals = []
    processed = []
    labels = []

    for path in selected_files:
        base_id = get_base_id(path)
        if base_id not in labels_map:
            continue

        img = Image.open(path).convert("RGB")
        originals.append(img)
        processed.append(transform(img))
        labels.append(id2label[labels_map[base_id]])

    n = len(originals)
    plt.figure(figsize=(3 * n, 6))

    for i in range(n):
        plt.subplot(2, n, i + 1)
        plt.imshow(originals[i])
        plt.title(labels[i])
        plt.axis("off")

        plt.subplot(2, n, i + 1 + n)
        plt.imshow(processed[i])
        plt.title(labels[i])
        plt.axis("off")

    plt.tight_layout()

    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path)
    plt.close()


save_clahe_samples(
    dataset_dir=train_dir,
    csv_path=train_label_path,
    out_dir=OUT_DIR,
    filename="clahe_samples.png",
    n_samples=20
)

In [18]:
def extract_wbc_crop2(
    img_rgb: np.ndarray,
    pad: int = 20,
    min_area_frac: float = 0.0005,
    q: float = 0.92,
    resize_mode: str = "resize",
    output_size: Tuple[int, int] = (224, 224),
) -> np.ndarray:
    def apply_resize(img: np.ndarray) -> np.ndarray:
        if resize_mode == "none":
            return img

        if resize_mode == "resize":
            return cv2.resize(img, output_size, interpolation=cv2.INTER_LINEAR)

        if resize_mode == "pad":
            target_h, target_w = output_size
            h, w = img.shape[:2]
            scale = min(target_w / w, target_h / h)
            new_w = max(1, int(round(w * scale)))
            new_h = max(1, int(round(h * scale)))

            img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

            pad_top = (target_h - new_h) // 2
            pad_bottom = target_h - new_h - pad_top
            pad_left = (target_w - new_w) // 2
            pad_right = target_w - new_w - pad_left

            return cv2.copyMakeBorder(
                img_resized,
                pad_top,
                pad_bottom,
                pad_left,
                pad_right,
                borderType=cv2.BORDER_CONSTANT,
                value=[0, 0, 0],
            )

        raise ValueError(f"Unsupported resize_mode: {resize_mode}")

    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    H = hsv[:, :, 0]
    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    hdist = np.minimum(np.abs(H - 140.0), 180.0 - np.abs(H - 140.0)) / 90.0
    hscore = 1.0 - np.clip(hdist, 0.0, 1.0)
    sscore = np.clip((S - 30.0) / 225.0, 0.0, 1.0)
    vscore = 1.0 - np.clip(V / 255.0, 0.0, 1.0)

    score = 0.55 * hscore + 0.25 * sscore + 0.20 * vscore
    thr = float(np.quantile(score, q))
    nuc = (score >= thr).astype(np.uint8)

    nuc = cv2.medianBlur(nuc * 255, 5)
    nuc = (nuc > 0).astype(np.uint8)

    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    nuc = cv2.morphologyEx(nuc, cv2.MORPH_OPEN, k1, iterations=1)
    nuc = cv2.morphologyEx(nuc, cv2.MORPH_CLOSE, k1, iterations=2)

    num, lab, stats, _ = cv2.connectedComponentsWithStats(nuc.astype(np.uint8), connectivity=8)
    if num <= 1:
        return apply_resize(img_rgb)

    h, w = nuc.shape
    min_area = int(min_area_frac * h * w)

    cx0, cy0 = w * 0.5, h * 0.5
    best_i = None
    best_score = -1e18
    for i in range(1, num):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        cx = float(stats[i, cv2.CC_STAT_LEFT] + 0.5 * stats[i, cv2.CC_STAT_WIDTH])
        cy = float(stats[i, cv2.CC_STAT_TOP] + 0.5 * stats[i, cv2.CC_STAT_HEIGHT])
        d2 = (cx - cx0) ** 2 + (cy - cy0) ** 2
        s = np.log1p(area) - 0.0008 * d2
        if s > best_score:
            best_score = s
            best_i = i

    if best_i is None:
        return apply_resize(img_rgb)

    nuc = (lab == best_i).astype(np.uint8)

    kbig = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (45, 45))
    mask = cv2.dilate(nuc, kbig, iterations=1)

    ys, xs = np.where(mask > 0)
    if xs.size == 0:
        return apply_resize(img_rgb)

    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(img_rgb.shape[1], x2 + pad)
    y2 = min(img_rgb.shape[0], y2 + pad)

    if (x2 - x1) < 10 or (y2 - y1) < 10:
        return apply_resize(img_rgb)

    crop = img_rgb[y1:y2, x1:x2]
    return apply_resize(crop)


In [19]:
def save_crop_samples(
    dataset_dir: str,
    csv_path: str,
    out_dir: str,
    filename: str = "crop_samples.png",
    n_samples: int = 5,
    resize_mode: str = "resize",
    output_size: Tuple[int, int] = (224, 224),
):
    os.makedirs(out_dir, exist_ok=True)

    files = get_filespath(dataset_dir)
    labels_map = get_labels(csv_path)
    id2label = {v: k for k, v in label2id.items()}

    selected_files = random.sample(files, min(n_samples, len(files)))

    originals = []
    processed = []
    labels = []

    for path in selected_files:
        base_id = get_base_id(path)
        if base_id not in labels_map:
            continue

        img = Image.open(path).convert("RGB")
        img_np = np.array(img)

        crop = extract_wbc_crop2(
            img_np,
            resize_mode=resize_mode,
            output_size=output_size,
        )

        originals.append(img)
        processed.append(crop)
        labels.append(id2label[labels_map[base_id]])

    n = len(originals)
    plt.figure(figsize=(3 * n, 6))

    for i in range(n):
        plt.subplot(2, n, i + 1)
        plt.imshow(originals[i])
        plt.title(labels[i])
        plt.axis("off")

        plt.subplot(2, n, i + 1 + n)
        plt.imshow(processed[i])
        plt.title(labels[i])
        plt.axis("off")

    plt.tight_layout()

    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path)
    plt.close()

In [20]:
save_crop_samples(
    dataset_dir=train_dir,
    csv_path=train_label_path,
    out_dir=OUT_DIR,
    filename="crop_samples_none.png",
    n_samples=10,
    resize_mode="none",
)

save_crop_samples(
    dataset_dir=train_dir,
    csv_path=train_label_path,
    out_dir=OUT_DIR,
    filename="crop_samples_resize.png",
    n_samples=10,
    resize_mode="resize",
    output_size=(224, 224),
)

save_crop_samples(
    dataset_dir=train_dir,
    csv_path=train_label_path,
    out_dir=OUT_DIR,
    filename="crop_samples_pad.png",
    n_samples=10,
    resize_mode="pad",
    output_size=(224, 224),
)